# 👥 EduNilai — Demographic Gap Analysis

**Stage:** 04 — Demographic Disaggregation
**Questions:**
- Does gender affect graduate employment outcomes?
- Does geography determine how much a degree pays back?
- Is income inequality narrowing or widening?


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 10,
})

BLUE   = '#2563EB'
ORANGE = '#EA580C'
GREEN  = '#16A34A'
RED    = '#DC2626'
GRAY   = '#6B7280'
PURPLE = '#7C3AED'

import os

def load(clean_name, raw_path):
    p = f'../data/cleaned/{clean_name}'
    return pd.read_csv(p if os.path.exists(p) else f'../data/{raw_path}', parse_dates=['date'])

lfs      = load('lfs_annual_sex.csv',                 'salary/lfs_annual_sex.csv')
sru_sex  = load('graduate_underemployment_sex.csv',   'salary/graduate_underemployment_sex.csv')
hh_nat   = load('hh_income_national.csv',             'demographic/hh_income_national.csv')
hh_state = load('hh_income_state.csv',                'demographic/hh_income_state.csv')
hh_pct   = load('hh_income_percentiles.csv',          'demographic/hh_income_percentiles.csv')
gini     = load('income_inequality_gini.csv',          'demographic/income_inequality_gini.csv')

lfs['sex'] = lfs['sex'].replace({'both': 'overall'})
lfs['year'] = lfs['date'].dt.year
if 'sru_person' in sru_sex.columns:
    sru_sex = sru_sex.rename(columns={'sru_person': 'sru_persons'})
sru_sex['year'] = sru_sex['date'].dt.year
hh_nat['year']   = hh_nat['date'].dt.year
hh_state['year'] = hh_state['date'].dt.year
hh_pct['year']   = hh_pct['date'].dt.year

print("All data loaded.")


---
## 👨‍👩 D1 — Labour Force Participation & Unemployment by Sex

In [ ]:
lfs_m = lfs[lfs['sex'] == 'male'].copy()
lfs_f = lfs[lfs['sex'] == 'female'].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Participation rate
axes[0].plot(lfs_m['year'], lfs_m['p_rate'], color=BLUE,   marker='o', markersize=5, linewidth=2, label='Male')
axes[0].plot(lfs_f['year'], lfs_f['p_rate'], color=ORANGE, marker='o', markersize=5, linewidth=2, label='Female')
axes[0].fill_between(lfs_m['year'], lfs_m['p_rate'].values, lfs_f['p_rate'].values, alpha=0.08, color=GRAY)
axes[0].set_title('Labour Force Participation Rate by Sex (%)', fontweight='bold')
axes[0].set_ylabel('Participation Rate (%)')
axes[0].set_xlabel('Year')
axes[0].legend()

# Participation gap
gap_p = lfs_m.set_index('year')['p_rate'] - lfs_f.set_index('year')['p_rate']
axes[1].bar(gap_p.index, gap_p.values, color=PURPLE, alpha=0.8)
axes[1].set_title('Male-Female Participation Rate Gap (pp)', fontweight='bold')
axes[1].set_ylabel('Percentage Points')
axes[1].set_xlabel('Year')
for yr, val in gap_p.items():
    axes[1].text(yr, val + 0.2, f'{val:.1f}', ha='center', fontsize=8)

plt.tight_layout()
plt.show()

print(f"Latest year ({lfs['year'].max()}) participation gap: {gap_p.iloc[-1]:.1f} pp")
print(f"  Male {lfs_m.iloc[-1]['p_rate']:.1f}% vs Female {lfs_f.iloc[-1]['p_rate']:.1f}%")


---
## 📉 D2 — Unemployment Rate by Sex

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(lfs_m['year'], lfs_m['u_rate'], color=BLUE,   marker='o', markersize=5, linewidth=2, label='Male')
axes[0].plot(lfs_f['year'], lfs_f['u_rate'], color=ORANGE, marker='o', markersize=5, linewidth=2, label='Female')
axes[0].axvspan(2020, 2021, alpha=0.1, color=RED)
axes[0].annotate('COVID-19', xy=(2020.5, max(lfs_f['u_rate'])*0.97),
                 ha='center', fontsize=8, color=RED, alpha=0.8)
axes[0].set_title('Unemployment Rate by Sex (%)', fontweight='bold')
axes[0].set_ylabel('Unemployment Rate (%)')
axes[0].set_xlabel('Year')
axes[0].legend()

# Female unemployment premium
gap_u = lfs_f.set_index('year')['u_rate'] - lfs_m.set_index('year')['u_rate']
colors_gap = [RED if v > 0 else GREEN for v in gap_u.values]
bars = axes[1].bar(gap_u.index, gap_u.values, color=colors_gap, alpha=0.8)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('Female Unemployment Premium over Male (pp)\n(Positive = Female worse off)',
                  fontweight='bold')
axes[1].set_ylabel('Percentage Points')
axes[1].set_xlabel('Year')
for bar, val in zip(bars, gap_u.values):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 val + (0.05 if val >= 0 else -0.12),
                 f'{val:+.2f}', ha='center', fontsize=8)

plt.tight_layout()
plt.show()


---
## 🎓 D3 — Graduate Underemployment (SRU) by Sex (2017–2024)

In [ ]:
sru_ann = sru_sex.groupby(['year','sex'])[['sru_rate','sru_persons']].mean().reset_index()

START, END = 2017, 2024
sru_f = sru_ann[(sru_ann['sex']=='female') & (sru_ann['year']>=START) & (sru_ann['year']<=END)]
sru_m = sru_ann[(sru_ann['sex']=='male')   & (sru_ann['year']>=START) & (sru_ann['year']<=END)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(sru_f['year'], sru_f['sru_rate'], color=ORANGE, marker='o', markersize=5, linewidth=2, label='Female')
axes[0].plot(sru_m['year'], sru_m['sru_rate'], color=BLUE,   marker='o', markersize=5, linewidth=2, label='Male')
axes[0].fill_between(sru_f['year'], sru_f['sru_rate'].values, sru_m['sru_rate'].values,
                     alpha=0.12, color=ORANGE, label='Gender gap')
axes[0].axvspan(2020, 2021, alpha=0.1, color=RED)
axes[0].set_title('Graduate Underemployment Rate (SRU) by Sex (%)', fontweight='bold')
axes[0].set_ylabel('SRU Rate (%)')
axes[0].set_xlabel('Year')
axes[0].set_xticks(range(START, END+1))
axes[0].legend()

gap_sru = sru_f.set_index('year')['sru_rate'] - sru_m.set_index('year')['sru_rate']
bars = axes[1].bar(gap_sru.index, gap_sru.values,
                   color=[RED if v > 0 else GREEN for v in gap_sru.values], alpha=0.8)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('Female SRU Premium over Male (pp)\n(Positive = Female more underemployed)',
                  fontweight='bold')
axes[1].set_ylabel('Percentage Points')
axes[1].set_xlabel('Year')
axes[1].set_xticks(range(START, END+1))
for bar, val in zip(bars, gap_sru.values):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 val + (0.1 if val >= 0 else -0.4),
                 f'{val:+.1f}', ha='center', fontsize=8.5)

plt.tight_layout()
plt.show()

print("Female vs Male SRU gap by year:")
print(gap_sru.to_string())
print(f"\nPre-COVID avg gap  (2017-2019): {gap_sru[2017:2019].mean():+.2f} pp")
print(f"COVID period       (2020-2021): {gap_sru[2020:2021].mean():+.2f} pp")
print(f"Post-COVID         (2022-2024): {gap_sru[2022:2024].mean():+.2f} pp")


---
## 🗺️ D4 — Regional Income Inequality by State

In [ ]:
nat_medians = hh_nat.set_index('year')['income_median']
years       = sorted(hh_state['year'].unique())

fig, axes = plt.subplots(1, 2, figsize=(15, 7))

latest_yr = hh_state['year'].max()
df_lat    = hh_state[hh_state['year']==latest_yr].sort_values('income_median', ascending=True)
nat_med   = nat_medians[latest_yr]
colors_s  = [RED if v >= nat_med else BLUE for v in df_lat['income_median']]

bars = axes[0].barh(df_lat['state'], df_lat['income_median'], color=colors_s, alpha=0.85)
axes[0].axvline(nat_med, color='black', linewidth=1.5, linestyle='--',
                label=f'National median (RM {nat_med:,})')
axes[0].set_xlabel('Median Monthly HH Income (RM)')
axes[0].set_title(f'Median Household Income by State — {latest_yr}', fontweight='bold')
axes[0].legend(fontsize=8)
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'RM {x:,.0f}'))
for bar, val in zip(bars, df_lat['income_median']):
    axes[0].text(bar.get_width() + 80, bar.get_y() + bar.get_height()/2,
                 f'RM {val:,}', va='center', fontsize=8)

# Urban vs rural ratio over time
TOP    = ['W.P. Kuala Lumpur', 'Selangor']
BOTTOM = ['Kelantan', 'Kedah', 'Sabah']
ratio_rows = []
for yr in years:
    df_yr  = hh_state[hh_state['year']==yr]
    top_m  = df_yr[df_yr['state'].isin(TOP)]['income_median'].mean()
    bot_m  = df_yr[df_yr['state'].isin(BOTTOM)]['income_median'].mean()
    ratio_rows.append({'year': yr, 'ratio': top_m / bot_m})
ratio_df = pd.DataFrame(ratio_rows)

axes[1].plot(ratio_df['year'], ratio_df['ratio'], color=RED, marker='o', markersize=7, linewidth=2.5)
axes[1].set_title('Urban-Rural Income Ratio\n(KL+Selangor vs Kelantan+Kedah+Sabah)', fontweight='bold')
axes[1].set_ylabel('Income Ratio (x)')
axes[1].set_xlabel('Year')
for _, row in ratio_df.iterrows():
    axes[1].text(row['year'], row['ratio'] + 0.02, f"{row['ratio']:.2f}x", ha='center', fontsize=9)

plt.tight_layout()
plt.show()

print(f"\nHighest: {df_lat.iloc[-1]['state']} — RM {df_lat.iloc[-1]['income_median']:,}")
print(f"Lowest:  {df_lat.iloc[0]['state']}  — RM {df_lat.iloc[0]['income_median']:,}")
print(f"Ratio:   {df_lat.iloc[-1]['income_median']/df_lat.iloc[0]['income_median']:.2f}x")


---
## 📊 D5 — Income Distribution by Percentile & Gini Coefficient

In [ ]:
hh_pct_clean = hh_pct.dropna(subset=['income'])
pct_mean = hh_pct_clean[hh_pct_clean['variable']=='mean'].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

yr_colors = {2019: BLUE, 2022: ORANGE, 2024: RED}
for yr, grp in pct_mean.groupby('year'):
    axes[0].plot(grp['percentile'], grp['income'] / 1000,
                 color=yr_colors.get(yr, GRAY), linewidth=2, label=str(yr))
axes[0].set_title('Mean Income by Percentile Group', fontweight='bold')
axes[0].set_xlabel('Income Percentile')
axes[0].set_ylabel('Mean Monthly Income (RM thousands)')
axes[0].legend(title='Year')
axes[0].axvline(50, color='black', linewidth=0.8, linestyle='--', alpha=0.5)
axes[0].annotate('Median\n(P50)', xy=(50, pct_mean[pct_mean['year']==2022].iloc[49]['income']/1000),
                 xytext=(55, pct_mean[pct_mean['year']==2022].iloc[49]['income']/1000),
                 fontsize=8, arrowprops=dict(arrowstyle='->', color='black'))

axes[1].plot(gini['date'].dt.year, gini['gini'],
             color=RED, marker='o', markersize=7, linewidth=2.5)
axes[1].set_title('Income Inequality — Gini Coefficient', fontweight='bold')
axes[1].set_ylabel('Gini Coefficient')
axes[1].set_xlabel('Year')
axes[1].set_ylim(0.35, 0.46)
for _, row in gini.iterrows():
    axes[1].text(row['date'].year, row['gini'] + 0.003,
                 f"{row['gini']:.3f}", ha='center', fontsize=9)

plt.tight_layout()
plt.show()


---
## 📋 D6 — Key Demographic Findings

In [ ]:
print("=" * 60)
print("DEMOGRAPHIC GAP — KEY FINDINGS")
print("=" * 60)

latest_yr  = lfs['year'].max()
lfs_m_lat  = lfs[(lfs['sex']=='male')   & (lfs['year']==latest_yr)].iloc[0]
lfs_f_lat  = lfs[(lfs['sex']=='female') & (lfs['year']==latest_yr)].iloc[0]
sru_f_2024 = sru_ann[(sru_ann['sex']=='female') & (sru_ann['year']==2024)].iloc[0]
sru_m_2024 = sru_ann[(sru_ann['sex']=='male')   & (sru_ann['year']==2024)].iloc[0]
top_s = hh_state[hh_state['year']==2022].sort_values('income_median').iloc[-1]
bot_s = hh_state[hh_state['year']==2022].sort_values('income_median').iloc[0]

print(f"\nGENDER GAP ({latest_yr}):")
print(f"  Participation: Male {lfs_m_lat['p_rate']:.1f}% vs Female {lfs_f_lat['p_rate']:.1f}%  (gap {lfs_m_lat['p_rate']-lfs_f_lat['p_rate']:.1f} pp)")
print(f"  Unemployment:  Male {lfs_m_lat['u_rate']:.1f}% vs Female {lfs_f_lat['u_rate']:.1f}%")
print(f"\nGRADUATE UNDEREMPLOYMENT (2024):")
print(f"  Female SRU: {sru_f_2024['sru_rate']:.1f}%")
print(f"  Male SRU:   {sru_m_2024['sru_rate']:.1f}%")
print(f"  Gap:        {sru_f_2024['sru_rate'] - sru_m_2024['sru_rate']:+.1f} pp")
print(f"\nREGIONAL GAP (2022):")
print(f"  Highest: {top_s['state']} — RM {top_s['income_median']:,}")
print(f"  Lowest:  {bot_s['state']} — RM {bot_s['income_median']:,}")
print(f"  Ratio:   {top_s['income_median']/bot_s['income_median']:.2f}x")
print(f"\nGINI (2022): {gini[gini['date'].dt.year==2022]['gini'].values[0]:.3f}")
print("=" * 60)

os.makedirs('../data/analysis', exist_ok=True)
sru_ann.to_csv('../data/analysis/sru_by_sex_annual.csv', index=False)
hh_state.to_csv('../data/analysis/hh_income_state_clean.csv', index=False)
print("\nSaved outputs to ../data/analysis/")
